In [7]:
import requests
import pandas as pd
import os
import time

In [8]:
HMDA_BASE = "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv"
#Home Mortgage Disclosure Act

In [ ]:
def scrape_hmda_loans(
    states: list[str],
    years: list[int],
    actions_taken: list[int],
    output_dir: str = "../A. Data Pipeline/Data/bronze/"
) -> pd.DataFrame:
    """
    Scrape loan data from HMDA Data Browser API.
    Parameters
    ----------
    states : list[str]
        List of state codes (e.g., ["CA", "NY", "TX"]).
    years : list[int]
        List of years (e.g., [2022]).
    actions_taken : list[int]
        List of action types (1=Originated, 3=Denied).
    output_dir : str
        Output directory to save raw files.
    Returns
    -------
    pd.DataFrame
        DataFrame containing loan data.
    """
    os.makedirs(output_dir, exist_ok=True)
    all_frames = []
    for year in years:
        params = {
            "years": year,
            "states": ",".join(states),
            "actions_taken": ",".join(str(a) for a in actions_taken),
        }
        
        print(f"[SCRAPING] HMDA {year} | States: {states} | Actions: {actions_taken}")

        response = requests.get(HMDA_BASE, params=params, stream=True)
        response.raise_for_status()

        # Stream to temp file (to avoid memory issues with large files))

        temp_file = os.path.join(output_dir, f"hmda_{year}_temp.csv")
        with open(temp_file, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        df_year = pd.read_csv(temp_file, low_memory=False)
        df_year["scrape_year"] = year
        all_frames.append(df_year)

        print(f"  → {len(df_year):,} records scraped")
        time.sleep(1)  # Rate limiting
    df_all = pd.concat(all_frames, ignore_index=True)
    return df_all


In [10]:
# 3 biggest states
TARGET_STATES = ["CA", "TX", "FL"] #if you want to scrape more states, add them here [e.g., "NY", "IL", "PA"]
TARGET_YEARS = [2022] #if you want to scrape more years, add them here [e.g., 2021, 2020, 2019]
# actions_taken: 1 = Originated, 3 = Denied
TARGET_ACTIONS = [1, 3]
df_loans = scrape_hmda_loans(
    states=TARGET_STATES,
    years=TARGET_YEARS,
    actions_taken=TARGET_ACTIONS,
)

[SCRAPING] HMDA 2022 | States: ['CA', 'TX', 'FL'] | Actions: [1, 3]
  → 2,830,687 records scraped


In [11]:
print(f"\n{'='*60}")
print(f"Total records scraped: {len(df_loans):,}")
print(f"Columns: {len(df_loans.columns)}")
print(f"\nAction taken distribution:")
print(df_loans["action_taken"].value_counts())


Total records scraped: 2,830,687
Columns: 100

Action taken distribution:
action_taken
1    2099761
3     730926
Name: count, dtype: int64


In [ ]:
df_loans.to_csv("../A. Data Pipeline/Data/bronze/hmda_loans_raw.csv", index=False)
print("\n Bronze: hmda_loans_raw.csv saved.")


 Bronze: hmda_loans_raw.csv saved.
